# HAR Model Architectures

Three diverse neural network architectures for **Human Activity Recognition**, all accepting a flat 561-dim input and outputting logits over 6 activity classes.

| Model | Key Idea |
|---|---|
| **MLP** | Fully connected, fast, simple baseline |
| **CNN** | Treats 561 features as a 1-D signal; learns local patterns |
| **LSTM** | Treats features as a short sequence; learns temporal structure |

> Architectural diversity is the core motivation for the ensemble approach.

## 1. Imports & Constants

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict

N_FEATURES = 561
N_CLASSES  = 6

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"N_FEATURES={N_FEATURES}  N_CLASSES={N_CLASSES}")

PyTorch version : 2.11.0+cpu
Device          : cpu
N_FEATURES=561  N_CLASSES=6


## 2. MLP — Fully Connected Baseline

A 3-layer MLP with **BatchNorm** and **Dropout**. Fast to train and a reliable baseline.

In [2]:
class MLP(nn.Module):
    """
    3-layer MLP with BatchNorm and Dropout.
    Fast to train; good baseline.
    """
    def __init__(
        self,
        in_features: int   = N_FEATURES,
        hidden:      int   = 256,
        n_classes:   int   = N_CLASSES,
        dropout:     float = 0.3,
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Return penultimate layer activations (for attack model)."""
        for layer in list(self.net.children())[:-1]:
            x = layer(x)
        return x

## 3. CNN — 1-D Convolutional Network

Reshapes the 561-dim input to `(1, 561)` and applies temporal convolutions. Learns **local feature interactions** that the MLP misses.

In [3]:
class CNN(nn.Module):
    """
    1-D Convolutional network.
    Reshapes 561 → (1, 561) and applies temporal convolutions.
    Learns local feature interactions the MLP misses.

    Conv1 (k=7) → MaxPool → (64, 280)
    Conv2 (k=5) → MaxPool → (128, 140)
    Conv3 (k=3) → AdaptiveAvgPool → (64, 1)
    """
    def __init__(
        self,
        in_features: int   = N_FEATURES,
        n_classes:   int   = N_CLASSES,
        dropout:     float = 0.3,
    ):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                     # → (64, 280)

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),                     # → (128, 140)

            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),             # → (64, 1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)                       # (B, 1, 561)
        x = self.conv_block(x)
        return self.classifier(x)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        x = self.conv_block(x)
        return x.view(x.size(0), -1)

## 4. LSTM — Bidirectional Recurrent Network

Splits the 561 features into **9 timesteps of 63 features** (padded), then runs a bidirectional LSTM. Captures **long-range dependencies** across sensor windows.

In [4]:
class LSTMNet(nn.Module):
    """
    Bidirectional LSTM.
    Splits 561 features into 9 timesteps of 63 features (+ 3 padding).
    Captures long-range dependencies across sensor windows.
    """
    def __init__(
        self,
        in_features:   int   = N_FEATURES,
        seq_len:       int   = 9,
        hidden:        int   = 128,
        n_layers:      int   = 2,
        n_classes:     int   = N_CLASSES,
        dropout:       float = 0.3,
        bidirectional: bool  = True,
    ):
        super().__init__()
        self.seq_len = seq_len
        feat_per_step = (in_features + seq_len - 1) // seq_len  # ceil div → 63
        self.feat_per_step = feat_per_step
        self.pad_total = feat_per_step * seq_len - in_features  # padding needed

        self.lstm = nn.LSTM(
            input_size=feat_per_step,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        dirs = 2 if bidirectional else 1
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden * dirs, n_classes),
        )

    def _reshape(self, x: torch.Tensor) -> torch.Tensor:
        if self.pad_total > 0:
            x = F.pad(x, (0, self.pad_total))   # pad right
        return x.view(x.size(0), self.seq_len, self.feat_per_step)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self._reshape(x)
        out, (hn, _) = self.lstm(x)
        # Use final hidden state of all directions concatenated
        if self.lstm.bidirectional:
            h = torch.cat([hn[-2], hn[-1]], dim=1)
        else:
            h = hn[-1]
        return self.classifier(h)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self._reshape(x)
        _, (hn, _) = self.lstm(x)
        if self.lstm.bidirectional:
            return torch.cat([hn[-2], hn[-1]], dim=1)
        return hn[-1]

## 5. Model Factory

In [5]:
ARCHITECTURES = {"mlp": MLP, "cnn": CNN, "lstm": LSTMNet}

def build_model(name: str, **kwargs) -> nn.Module:
    """
    Instantiate a model by name with optional keyword overrides.

    Examples
    --------
    build_model("mlp")               → MLP with defaults
    build_model("cnn", dropout=0.4)  → CNN with custom dropout
    """
    name = name.lower()
    if name not in ARCHITECTURES:
        raise ValueError(f"Unknown model '{name}'. Choose from {list(ARCHITECTURES)}")
    return ARCHITECTURES[name](**kwargs)


def count_params(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 6. Sanity Check — Forward Pass & Feature Extraction

In [6]:
x = torch.randn(16, N_FEATURES)  # batch of 16 samples

print(f"{'Model':<8} {'Params':>12}  {'Output shape':<16}  {'Feature shape'}")
print("-" * 58)

for name in ["mlp", "cnn", "lstm"]:
    model = build_model(name)
    model.eval()
    with torch.no_grad():
        out   = model(x)
        feats = model.extract_features(x)
    print(f"{name.upper():<8} {count_params(model):>12,}  {str(tuple(out.shape)):<16}  {tuple(feats.shape)}")

print("\n✅ All architectures OK.")

Model          Params  Output shape      Feature shape
----------------------------------------------------------
MLP           178,310  (16, 6)           (16, 128)
CNN            67,142  (16, 6)           (16, 64)
LSTM          594,438  (16, 6)           (16, 256)

✅ All architectures OK.


## 7. Model Architecture Summary

In [7]:
# Print full architecture for each model
for name in ["mlp", "cnn", "lstm"]:
    model = build_model(name)
    print(f"{'='*50}")
    print(f" {name.upper()}  —  {count_params(model):,} trainable parameters")
    print(f"{'='*50}")
    print(model)
    print()

 MLP  —  178,310 trainable parameters
MLP(
  (net): Sequential(
    (0): Linear(in_features=561, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=6, bias=True)
  )
)

 CNN  —  67,142 trainable parameters
CNN(
  (conv_block): Sequential(
    (0): Conv1d(1, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=

## 8. Custom Model Example

Use `build_model()` with keyword arguments to override any default hyperparameter.

In [8]:
# Example: wider MLP with less dropout
mlp_wide = build_model("mlp", hidden=512, dropout=0.2)
print(f"Wide MLP params : {count_params(mlp_wide):,}")

# Example: CNN with more regularisation
cnn_reg = build_model("cnn", dropout=0.5)
print(f"Regularised CNN : {count_params(cnn_reg):,}")

# Example: unidirectional LSTM, single layer
lstm_light = build_model("lstm", hidden=64, n_layers=1, bidirectional=False)
print(f"Light LSTM      : {count_params(lstm_light):,}")

Wide MLP params : 422,150
Regularised CNN : 67,142
Light LSTM      : 33,414
